In [86]:
import java.time.LocalTime
import java.time.format.DateTimeFormatter
import kotlin.text.MatchResult
import kotlin.text.get

// ========== Domain model ==========
sealed class LogEvent {
    data class ConfirmationRuleUpdate(
        val slot: Long,
        val headSlot: Long,
        val headShortRootHex: String,           // e.g., 0x9acceb
        val confirmedSlot: Long,
        val confirmedShortRootHex: String,      // e.g., 0x6dba83
    ) : LogEvent()

    data class SlotEvent(
        val time: LocalTime,
        val slot: Long,
        val blockRootHex: String?,              // null when "... empty"
    ) : LogEvent()
    
    data class NewVotestInBlock(
        val slot: Long,
        val newVotes: Int
    ) : LogEvent()

    data class Unknown(val line: String) : LogEvent()
}

// ========== Parser ==========
object TekuLogParser {
    private val timeFmt = DateTimeFormatter.ofPattern("HH:mm:ss.SSS")

    // 1) updateConfirmationRuleStore
    private val reUpdate = Regex(
        """
        ^updateConfirmationRuleStore:\s*
        slot=(?<slot>\d+),\s*
        head=(?<head>\d+),\(
            (?<headHex>0x[0-9a-fA-F]+)
        \),\s*
        confirmed=(?<conf>\d+)\(
            (?<delta>-?\d+)
        \),\(
            (?<confHex>0x[0-9a-fA-F]+)
        \),\s*
        states\s+requested/uniq:\s*
            (?<req>\d+)/
            (?<uniq>\d+),\s*
        justified=Checkpoint\[
            (?<jEpoch>\d+),\s*
            (?<jHex>0x[0-9a-fA-F]+)
        \]\s*in\s*
            (?<ms>\d+)\s*ms
        $
        """.trimIndent().compactWs(),
        setOf()
    )

    // 2a) Full Slot Event
    private val reSlotFull = Regex(
        """
        ^(?<time>\d{2}:\d{2}:\d{2}\.\d{3})\s+(?<lvl>\S+)\s+-\s+
        Slot\s+Event\s+\*{3}\s+
        Slot:\s*(?<slot>\d+),\s*
        Block:\s*(?<block>[0-9a-fA-F]{64}),\s*
        Justified:\s*(?<j>\d+),\s*
        Finalized:\s*(?<f>\d+),\s*
        Peers:\s*(?<peers>\d+)
        $
        """.trimIndent().compactWs(),
        setOf(RegexOption.IGNORE_CASE)
    )

    // 2b) Short/empty Slot Event (… empty and the rest may be ellipsis)
    private val reSlotEmpty = Regex(
        """
        ^(?<time>\d{2}:\d{2}:\d{2}\.\d{3})\s+(?<lvl>\S+)\s+-\s+
        Slot\s+Event\s+\*{3}\s+
        Slot:\s*(?<slot>\d+),\s*
        Block:\s*\.{3}\s*empty
        (?:,.*)?$
        """.trimIndent().compactWs(),
        setOf(RegexOption.IGNORE_CASE)
    )

    private val newVotesInBlock = Regex(
        """
        ^Importing\sblock:\s(?<slot>\d+),.+\svotes\sin\sblock:\s(?<votes>\d+)\s*$
        """.trimIndent().compactWs(),
        setOf(RegexOption.IGNORE_CASE)
    )
    
    
    fun parse(line: String): LogEvent {
        reUpdate.matchEntire(line)?.let { m ->
            return LogEvent.ConfirmationRuleUpdate(
                slot = m.group("slot").toLong(),
                headSlot = m.group("head").toLong(),
                headShortRootHex = m.group("headHex"),
                confirmedSlot = m.group("conf").toLong(),
                confirmedShortRootHex = m.group("confHex"),
            )
        }

        reSlotFull.matchEntire(line)?.let { m ->
            return LogEvent.SlotEvent(
                time = LocalTime.parse(m.group("time"), timeFmt),
                slot = m.group("slot").toLong(),
                blockRootHex = m.group("block")
            )
        }

        reSlotEmpty.matchEntire(line)?.let { m ->
            return LogEvent.SlotEvent(
                time = LocalTime.parse(m.group("time"), timeFmt),
                slot = m.group("slot").toLong(),
                blockRootHex = null,            // ... empty
            )
        }

        newVotesInBlock.matchEntire(line)?.let { m ->
            return LogEvent.NewVotestInBlock(
                slot = m.group("slot").toLong(),
                newVotes = m.group("votes").toInt()
            )
        }

        return LogEvent.Unknown(line)
    }

    // --- helpers ---

    private fun MatchGroupCollection.getByName(name: String) =
        (this as MatchNamedGroupCollection).get(name)

    private fun MatchResult.group(name: String): String =
        this.groups.getByName(name)?.value ?: error("Missing group '$name'")

    // Tolerate varied whitespace without making the regex unreadable
    private fun String.compactWs(): String =
        replace("\n", "").replace(Regex("\\s+"), "\\s*")

    // "arrival 8760ms, gossip_validation +0ms, processed +85ms, ..."
    private fun parseTimings(raw: String): LinkedHashMap<String, Long> {
        val map = LinkedHashMap<String, Long>()
        val pair = Regex("""([a-zA-Z0-9_]+)\s+([+\-]?\d+)ms""")
        for (part in raw.split(Regex("""\s*,\s*"""))) {
            val m = pair.find(part) ?: continue
            val key = m.groupValues[1]
            val v = m.groupValues[2].toLong()
            map[key] = v
        }
        return map
    }
}

// ========== Quick demo ==========
val lines1 = listOf(
    "Importing block: 12657893, 0x82929bc039a20ce96ba5c4658fe4e289f72c980778966300c1d256276e96663d, votes in block: 66",
    "updateConfirmationRuleStore: slot=12657889, head=12657889,(0x929850), confirmed=12657888(-1),(0x3b1398), states requested/uniq: 2/1, justified=Checkpoint[395559, 0x3b1398] in 1 ms",
    // old log line
    "updateConfirmationRuleStore: head=12657983,(0x9acceb), confirmed=12657982(-1),(0x6dba83), states requested/uniq: 5/2, justified=Checkpoint[395560, 0xe30323] in 598 ms",
    "17:17:03.812 INFO  - Slot Event  *** Slot: 12657983, Block: 9accebd69baa3e90411f123121f9eb7d2fc61f8117d81accf87f81c7bafde347, Justified: 395560, Finalized: 395559, Peers: 63",
    "17:05:51.527 INFO  - Slot Event  *** Slot: 12657927, Block: ... empty,    Justified: ...",
    "17:17:11.002 INFO  - Epoch Event *** Epoch: 395562, Justified checkpoint: 395561, Finalized checkpoint: 395560, Finalized root: e3032378ff11e040a689579c8a3eefa4e45485ee2ff270709a431106aac568a7",
    "18:23:07.855 WARN  - Late Block Import *** Block: e930c41bbe0aa4e2ae3656ff4702f906d908d49592decd9da437def301954cb3 (12658313) Proposer: 3126 Result: success Timings: arrival 8760ms, gossip_validation +0ms, pre-state_retrieved +2ms, processed +85ms, data_availability_checked +0ms, execution_payload_result_received +0ms, begin_importing +0ms, transaction_prepared +0ms, transaction_committed +0ms, completed +8ms",
    "18:23:13.482 INFO  - Reorg Event *** New Head: 6804d44b51fd3b957c147298575f53e8929769dfba4b6d80ca4725a08b50176d (12658314), Previous Head: e930c41bbe0aa4e2ae3656ff4702f906d908d49592decd9da437def301954cb3 (12658313), Common Ancestor: 41b24017534a5266d112fea39e7f7e10c1fade19272ee7e427b42cd586e2853a (12658312)"
)
val parsed = lines1.map(TekuLogParser::parse)

parsed.joinToString("\n")


NewVotestInBlock(slot=12657893, newVotes=66)
ConfirmationRuleUpdate(slot=12657889, headSlot=12657889, headShortRootHex=0x929850, confirmedSlot=12657888, confirmedShortRootHex=0x3b1398)
Unknown(line=updateConfirmationRuleStore: head=12657983,(0x9acceb), confirmed=12657982(-1),(0x6dba83), states requested/uniq: 5/2, justified=Checkpoint[395560, 0xe30323] in 598 ms)
SlotEvent(time=17:17:03.812, slot=12657983, blockRootHex=9accebd69baa3e90411f123121f9eb7d2fc61f8117d81accf87f81c7bafde347)
SlotEvent(time=17:05:51.527, slot=12657927, blockRootHex=null)
Unknown(line=17:17:11.002 INFO  - Epoch Event *** Epoch: 395562, Justified checkpoint: 395561, Finalized checkpoint: 395560, Finalized root: e3032378ff11e040a689579c8a3eefa4e45485ee2ff270709a431106aac568a7)
Unknown(line=18:23:07.855 WARN  - Late Block Import *** Block: e930c41bbe0aa4e2ae3656ff4702f906d908d49592decd9da437def301954cb3 (12658313) Proposer: 3126 Result: success Timings: arrival 8760ms, gossip_validation +0ms, pre-state_retrieved +2

In [111]:
%use dataframe
import java.io.File

class ConfLogs(
    val logFile: String
) {

    val logEvents = File(logFile)
        .readLines()
        .map { TekuLogParser.parse(it) }
        .filter { it !is LogEvent.Unknown }

    val confirmLag: List<Pair<Long, Long>> = logEvents
        .filterIsInstance<LogEvent.ConfirmationRuleUpdate>()
        // leaving only events when confirmed slot advances 
//    .zipWithNext()
//    .mapNotNull { if (it.first.confirmedSlot == it.second.confirmedSlot) null else it.first }
        .map { it.slot to (it.slot - it.confirmedSlot) }

    val df = confirmLag.toDataFrame()
        .rename { all() }.into("slot", "confirm_lag")
        .inferType()
}


val runs = mapOf(
    "1. noOpts" to ConfLogs("./conf-sim-7-no-opt.log"),
    "2. emptySlotOptOnly" to ConfLogs("./conf-sim-7-empty-slot-opt.log"),
    "3. emptySlotAndLateBlockOpt" to ConfLogs("./conf-sim-6-late-block-opt.log"),
)

val runsDf = runs
    .map { (name, logs) ->
        logs.df.add("run") { name }
    }
    .reduce { a1, a2 -> a1.concat(a2)} 

In [88]:
%use kandy


In [113]:
import org.jetbrains.kotlinx.kandy.letsplot.layers.builders.subcontext.BorderLine


runsDf.plot {
    x(slot)
    y(confirm_lag)
    bars {
        this.borderLine {
            this.type = LineType.BLANK
            this.width = 0.0
        }
        fillColor = Color.BLUE
        
    }

    facetWrap(nCol = 1) {
        facet(run)
    }

    layout {
        size = 2500 to 2000
        style { 
            panel.grid.majorXLine { blank = true }
        }
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.3.3/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="V93Osd"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 var plotSpec={
"mapping":{
},
"data":{
"run":["1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts","1. noOpts"

In [114]:
runs.map { (name, v) -> 
    val lags = v.confirmLag.map { it.second }
    val countMap = lags.groupingBy { it }.eachCount().toSortedMap()
    name + ":\n"+ countMap
}.joinToString("\n")    

1. noOpts:
{1=7489, 2=343, 3=80, 4=63, 5=22, 6=1, 7=1}
2. emptySlotOptOnly:
{1=7535, 2=345, 3=80, 4=23, 5=15, 6=1}
3. emptySlotAndLateBlockOpt:
{1=7705, 2=225, 3=64, 4=5}

In [108]:
val logs = runs.entries.first().value
val updEvents = logs.logEvents
    .filterIsInstance<LogEvent.ConfirmationRuleUpdate>()
val blockCount = updEvents
    .map { it.headSlot }
    .distinct()
    .count()
val slotCount = updEvents.last().slot - updEvents.first().slot

"Empty slots count: " + (slotCount - blockCount)

Empty slots count: 45

In [90]:

val lags = df1.getColumn { confirm_lag }.toList()
val map = lags.groupingBy { it }.eachCount().toSortedMap()

val totCnt = map.values.sum()
val percentMap = map.mapValues { it.value.toDouble() * 100 / totCnt }

val dfMap = mapOf(
    "conf_distance" to percentMap.keys.toList(),
    "fraction" to percentMap.values.toList(),
)

plot(dfMap) {
    bars {
        x(percentMap.keys) {
            axis {
                name = "Confirmation distance in slots"
            }
        }
        y(percentMap.values) {
            axis {
                name = "Fraction in %%"
            }
        }
    }
}


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.3.3/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="6rw8Zv"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 var plotSpec={
"mapping":{
},
"data":{
"x":[1.0,2.0,3.0,4.0,5.0,6.0],
"y":[94.3508156936767,4.314412835378185,0.9302952676284212,0.24268572199002292,0.14830794121612512,0.013482540110556829]
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"Confirmation distance in slots",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"Fraction in %%",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"sampling":"none",
"position":"dodge",
"geom":"bar",
"data":{
}
}]
};
 var plotContainer = document.getElementById("6rw8Zv");
 LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer);
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 1 
 
 
 
 
 
 
 
 
 2 
 
 
 
 
 
 
 
 
 3 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 60 
 
 
 
 
 
 
 70 
 
 
 
 
 
 
 80 
 
 
 
 
 
 
 90 
 
 
 
 
 
 
 
 
 Fraction in %% 
 
 
 
 
 Confirmation distance in slots

In [91]:
map

{1=6998, 2=320, 3=69, 4=18, 5=11, 6=1}

In [92]:
import org.jetbrains.letsPlot.core.spec.back.transform.bistro.util.scale

plot { 
    histogram(x = df1.getColumn { confirm_lag }) {
        y { 
            scale = continuous(transform = Transformation.LOG2)
        }
    } 
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.3.3/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="sVBiCI"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 var plotSpec={
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"confirm_lag",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null],
"trans":"LOG2"
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"count"
},
"stat":"identity",
"data":{
"count":[6998.0,0.0,0.0,0.0,320.0,0.0,0.0,0.0,69.0,0.0,0.0,0.0,18.0,0.0,0.0,0.0,11.0,0.0,0.0,1.0],
"x":[1.125,1.375,1.625,1.875,2.125,2.375,2.625,2.875,3.125,3.375,3.625,3.875,4.125,4.375,4.625,4.875,5.125,5.375,5.625,5.875]
},
"sampling":"none",
"position":"identity",
"geom":"bar"
}]
};
 var plotContainer = document.getElementById("sVBiCI");
 LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer);
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 1 
 
 
 
 
 
 
 
 
 2 
 
 
 
 
 
 
 
 
 3 
 
 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 
 
 
 
 1 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 16 
 
 
 
 
 
 
 64 
 
 
 
 
 
 
 256 
 
 
 
 
 
 
 1,024 
 
 
 
 
 
 
 4,096 
 
 
 
 
 
 
 
 
 count 
 
 
 
 
 confirm_lag

In [93]:
val newVotesInBlock = logEvents
    .filterIsInstance<LogEvent.NewVotestInBlock>()
    .map { it.newVotes }
    .filter { it < 2000 }

plot {
    histogram(newVotesInBlock) {
//        y {
//            scale = continuous(transform = Transformation.LOG2)
//        }
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.3.3/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="ILqbhL"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 var plotSpec={
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"x",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"count"
},
"stat":"identity",
"data":{
"count":[1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0],
"x":[13.425,40.275000000000006,67.125,93.97500000000001,120.825,147.675,174.52500000000003,201.375,228.22500000000002,255.075,281.925,308.77500000000003,335.625,362.475,389.32500000000005,416.175,443.02500000000003,469.875,496.725,523.575]
},
"sampling":"none",
"position":"identity",
"geom":"bar"
}]
};
 var plotContainer = document.getElementById("ILqbhL");
 LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer);
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 
 
 300 
 
 
 
 
 
 
 
 
 400 
 
 
 
 
 
 
 
 
 500 
 
 
 
 
 
 
 
 
 
 
 0.0 
 
 
 
 
 
 
 0.5 
 
 
 
 
 
 
 1.0 
 
 
 
 
 
 
 1.5 
 
 
 
 
 
 
 2.0 
 
 
 
 
 
 
 
 
 count 
 
 
 
 
 x

In [94]:
"" + newVotesInBlock.sum() + " in " + newVotesInBlock.size + " blocks, average: " + (newVotesInBlock.sum() / newVotesInBlock.size)

1062 in 3 blocks, average: 354